# Baseline 1b: Keyword Search Using Okapi BM25 Model

## 1. Advanced Mathematical Theory (For Q&A / Defense)

Okapi BM25 is an upgraded algorithm based on the foundation of TF-IDF, widely considered the industry standard for keyword-based text retrieval systems (Lexical Search).

### Okapi BM25 Scoring Formula:
$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

#### Key Components in the Formula:
1. **$f(q_i, D)$**: The frequency of query term $q_i$ in document $D$ (Term Frequency).
2. **BM25 IDF**:
   $$\text{IDF}(q_i) = \log \frac{|D| - n(q_i) + 0.5}{n(q_i) + 0.5}$$
   *(Where $n(q_i)$ is the number of documents containing term $q_i$. If a keyword appears in more than half of the documents, the BM25 IDF can become negative - which is a characteristic feature of this formula).* 

3. **Term Frequency Saturation ($k_1$)**:
   - The parameter $k_1$ (typically chosen between $1.2$ and $2.0$) controls the saturation limit.
   - In TF-IDF, the score grows linearly with term frequency. In BM25, when a term appears too many times, the score asymptotically approaches a saturation limit to prevent abnormally repetitive words from manipulating search results.

4. **Document Length Normalization ($b$)**:
   - The parameter $b$ (typically chosen as $0.75$) controls the penalty for long documents.
   - $|D| / \text{avgdl}$ is the ratio of the document's length compared to the average length across the corpus.
   - If a document is very long ($|D| \gg \text{avgdl}$), the algorithm penalizes its score because the matching keywords might have occurred by chance. If a document is short and concise ($|D| \ll \text{avgdl}$), matching keywords are rewarded more heavily.

## 2. Initialize Toy Corpus (Sample Data)
We reuse the same sample corpus from the TF-IDF notebook to easily compare results.

In [1]:
import numpy as np
import pandas as pd
import re

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))

print(f"[Log] Number of documents: {len(tokenized_docs)}")
print(f"[Log] Total vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")

[Log] Number of documents: 5
[Log] Total vocabulary size: 28
Vocabulary: ['$10959', '$29760', '$383285', '$52729', '$574785', '2023', '2024', 'acquisition', 'amazon', 'and', 'apple', 'equipment', 'fiscal', 'for', 'in', 'income', 'million', 'net', 'nvidia', 'of', 'payments', 'plant', 'property', 'purchases', 'sales', 'was', 'were', 'year']


## 3. Manual BM25 Indexing (Step-by-Step BM25 Index Building)
We calculate the average statistics of the corpus to plug into the formula.

In [2]:
# 3.1 Calculate document lengths and average document length (avgdl)
doc_lengths = {k: len(v) for k, v in tokenized_docs.items()}
avgdl = sum(doc_lengths.values()) / len(doc_lengths)

print("=== DOCUMENT LENGTH STATISTICS ===")
for k, l in doc_lengths.items():
    print(f"  - {k}: {l} words (Ratio length/avgdl = {l/avgdl:.2f})")
print(f"\nAverage corpus length (avgdl): {avgdl:.2f} words")

=== DOCUMENT LENGTH STATISTICS ===
  - doc1: 10 words (Ratio length/avgdl = 0.91)
  - doc2: 12 words (Ratio length/avgdl = 1.09)
  - doc3: 10 words (Ratio length/avgdl = 0.91)
  - doc4: 13 words (Ratio length/avgdl = 1.18)
  - doc5: 10 words (Ratio length/avgdl = 0.91)

Average corpus length (avgdl): 11.00 words


### 3.2 Compute BM25 IDF Matrix
Note that the BM25 IDF formula differs from the standard TF-IDF formula.

In [3]:
bm25_idf = {}
num_docs = len(corpus)

for word in vocabulary:
    # n(q_i): number of documents containing the word
    n_qi = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    # Calculate IDF using the BM25 formula
    idf = np.log((num_docs - n_qi + 0.5) / (n_qi + 0.5) + 1.0) # Avoid negative idf
    bm25_idf[word] = idf

df_bm25_idf = pd.Series(bm25_idf).sort_values(ascending=False)
print("[SUCCESS] BM25 IDF calculation complete!")
print("\n=== COMPLETE BM25 IDF OF VOCABULARY ===")
df_bm25_idf_df = pd.DataFrame({'Word': df_bm25_idf.index, 'IDF': df_bm25_idf.values})
print(df_bm25_idf_df.to_string(index=False))
df_bm25_idf_df

[SUCCESS] BM25 IDF calculation complete!

=== COMPLETE BM25 IDF OF VOCABULARY ===
       Word      IDF
     $10959 1.386294
     $29760 1.386294
    $383285 1.386294
     $52729 1.386294
    $574785 1.386294
       2024 1.386294
acquisition 1.386294
        was 1.386294
   payments 1.386294
     nvidia 1.386294
     income 1.386294
        for 1.386294
      plant 1.386294
  purchases 1.386294
   property 0.875469
      apple 0.875469
     amazon 0.875469
  equipment 0.875469
        and 0.875469
         of 0.875469
      sales 0.875469
       2023 0.538997
        net 0.538997
     fiscal 0.287682
       were 0.287682
         in 0.287682
       year 0.287682
    million 0.087011


,Word,IDF
0,$10959,1.386294
1,$29760,1.386294
2,$383285,1.386294
3,$52729,1.386294
4,$574785,1.386294
5,2024,1.386294
6,acquisition,1.386294
7,was,1.386294
8,payments,1.386294
9,nvidia,1.386294


### 3.3 Construct the Complete BM25 Weight Matrix (Document-Term Weight Matrix)
Similar to TF-IDF, we can precompute the BM25 weights for every pair of (document, term) in the vocabulary to get an overview of the weight matrix.

In [4]:
bm25_matrix = []
k1 = 1.2
b = 0.75
for doc_name, tokens in tokenized_docs.items():
    doc_len = len(tokens)
    doc_weights = {}
    for word in vocabulary:
        tf = tokens.count(word)
        if tf > 0:
            idf = bm25_idf[word]
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avgdl))
            weight = idf * (numerator / denominator)
        else:
            weight = 0.0
        doc_weights[word] = weight
    bm25_matrix.append(doc_weights)

df_bm25 = pd.DataFrame(bm25_matrix, index=corpus.keys())
print("[SUCCESS] BM25 weight matrix built successfully!")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n=== COMPLETE BM25 WEIGHT MATRIX ===")
print(df_bm25.to_string())
df_bm25

[SUCCESS] BM25 weight matrix built successfully!

=== COMPLETE BM25 WEIGHT MATRIX ===
        $10959    $29760   $383285   $52729   $574785      2023      2024  acquisition    amazon       and     apple  equipment    fiscal       for        in    income   million       net    nvidia        of  payments     plant  property  purchases     sales       was      were      year
doc1  0.000000  0.000000  1.439842  0.00000  0.000000  0.559816  0.000000     0.000000  0.000000  0.000000  0.909285   0.000000  0.298794  0.000000  0.298794  0.000000  0.090372  0.559816  0.000000  0.000000  0.000000  0.000000  0.000000    0.00000  0.909285  0.000000  0.298794  0.298794
doc2  1.336587  0.000000  0.000000  0.00000  0.000000  0.000000  0.000000     1.336587  0.000000  0.844077  0.844077   0.844077  0.000000  1.336587  0.000000  0.000000  0.083891  0.000000  0.000000  0.844077  1.336587  1.336587  0.844077    0.00000  0.000000  0.000000  0.277367  0.000000
doc3  0.000000  0.000000  0.000000  0.00000  1.

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.000000,1.439842,0.00000,0.000000,0.559816,0.000000,0.000000,0.000000,0.000000,0.909285,0.000000,0.298794,0.000000,0.298794,0.000000,0.090372,0.559816,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.909285,0.000000,0.298794,0.298794
doc2,1.336587,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.336587,0.000000,0.844077,0.844077,0.844077,0.000000,1.336587,0.000000,0.000000,0.083891,0.000000,0.000000,0.844077,1.336587,1.336587,0.844077,0.00000,0.000000,0.000000,0.277367,0.000000
doc3,0.000000,0.000000,0.000000,0.00000,1.439842,0.559816,0.000000,0.000000,0.909285,0.000000,0.000000,0.000000,0.298794,0.000000,0.298794,0.000000,0.090372,0.559816,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.909285,0.000000,0.298794,0.298794
doc4,0.000000,0.000000,0.000000,1.29032,0.000000,0.501681,0.000000,0.000000,0.814859,0.814859,0.000000,0.814859,0.267766,0.000000,0.267766,0.000000,0.080988,0.000000,0.000000,0.814859,0.000000,0.000000,0.814859,1.29032,0.000000,0.000000,0.267766,0.267766
doc5,0.000000,1.439842,0.000000,0.00000,0.000000,0.000000,1.439842,0.000000,0.000000,0.000000,0.000000,0.000000,0.298794,0.000000,0.298794,1.439842,0.090372,0.559816,1.439842,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,1.439842,0.000000,0.298794


## 4. Execute Search and BM25 Scoring
Implement a function to compute the BM25 score for each document with standard parameters $k_1 = 1.2$ and $b = 0.75$.

In [5]:
def search_bm25(query, k1=1.2, b=0.75):
    print(f"\n=== USER QUERY: '{query}' ===")
    query_tokens = tokenize(query)
    print(f"[Step 1] Tokenized Query: {query_tokens}")
    
    scores = {}
    for doc_name, tokens in tokenized_docs.items():
        score = 0.0
        doc_len = len(tokens)
        print(f"\n* Calculating BM25 score for {doc_name} (Length L_d = {doc_len} words, avgdl = {avgdl:.2f}):")
        
        for word in query_tokens:
            if word in vocabulary:
                tf = tokens.count(word)
                idf = bm25_idf[word]
                
                # Calculate each part of the BM25 formula in detail
                ratio = doc_len / avgdl
                len_penalty = 1 - b + b * ratio
                numerator = tf * (k1 + 1)
                denominator = tf + k1 * len_penalty
                term_score = idf * (numerator / denominator)
                
                score += term_score
                print(f"  - Word '{word}': tf = {tf}, idf = {idf:.4f}")
                print(f"    + Document length ratio L_d / avgdl = {doc_len} / {avgdl:.2f} = {ratio:.4f}")
                print(f"    + Length penalty (1 - b + b * (L_d/avgdl)) = 1 - {b} + {b} * {ratio:.4f} = {len_penalty:.4f}")
                print(f"    + Numerator (tf * (k1 + 1)) = {tf} * {k1+1:.1f} = {numerator:.4f}")
                print(f"    + Denominator (tf + k1 * penalty) = {tf} + {k1} * {len_penalty:.4f} = {denominator:.4f}")
                print(f"    + Term score = idf * (num / den) = {idf:.4f} * ({numerator:.4f} / {denominator:.4f}) = {term_score:.4f}")
            else:
                print(f"  - Word '{word}': Not in vocabulary (OOV)")
                
        scores[doc_name] = score
        print(f"  => TOTAL BM25 SCORE FOR doc5 = {score:.4f}")
        
    # Sort and rank
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\n=== BM25 RETRIEVAL RANKING RESULTS ===")
    for rank, (doc_name, score) in enumerate(sorted_scores):
        print(f"  Rank {rank+1}: {doc_name} | BM25 Score: {score:.4f} | Content: {corpus[doc_name]}")
        
    return sorted_scores

## 5. Testing and Synonym Comparison (Failure Analysis)

Now, we run two queries to compare.

In [6]:
# Test 1: Keyword Match
search_bm25("Apple Net sales in 2023")


=== USER QUERY: 'Apple Net sales in 2023' ===
[Step 1] Tokenized Query: ['apple', 'net', 'sales', 'in', '2023']

* Calculating BM25 score for doc1 (Length L_d = 10 words, avgdl = 11.00):
  - Word 'apple': tf = 1, idf = 0.8755
    + Document length ratio L_d / avgdl = 10 / 11.00 = 0.9091
    + Length penalty (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Numerator (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Denominator (tf + k1 * penalty) = 1 + 1.2 * 0.9318 = 2.1182
    + Term score = idf * (num / den) = 0.8755 * (2.2000 / 2.1182) = 0.9093
  - Word 'net': tf = 1, idf = 0.5390
    + Document length ratio L_d / avgdl = 10 / 11.00 = 0.9091
    + Length penalty (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Numerator (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Denominator (tf + k1 * penalty) = 1 + 1.2 * 0.9318 = 2.1182
    + Term score = idf * (num / den) = 0.5390 * (2.2000 / 2.1182) = 0.5598
  - Word 'sales': tf = 1, idf = 0.8755
    + Document length r

[('doc1', np.float64(3.236996724322915)),
 ('doc3', np.float64(2.3277115979725123)),
 ('doc5', np.float64(0.858610363564984)),
 ('doc2', np.float64(0.8440774280463896)),
 ('doc4', np.float64(0.7694469796563125))]

In [7]:
# Test 2: Financial Synonym (Lexical Gap)
search_bm25("Amazon capital expenditures 2023")


=== USER QUERY: 'Amazon capital expenditures 2023' ===
[Step 1] Tokenized Query: ['amazon', 'capital', 'expenditures', '2023']

* Calculating BM25 score for doc1 (Length L_d = 10 words, avgdl = 11.00):
  - Word 'amazon': tf = 0, idf = 0.8755
    + Document length ratio L_d / avgdl = 10 / 11.00 = 0.9091
    + Length penalty (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Numerator (tf * (k1 + 1)) = 0 * 2.2 = 0.0000
    + Denominator (tf + k1 * penalty) = 0 + 1.2 * 0.9318 = 1.1182
    + Term score = idf * (num / den) = 0.8755 * (0.0000 / 1.1182) = 0.0000
  - Word 'capital': Not in vocabulary (OOV)
  - Word 'expenditures': Not in vocabulary (OOV)
  - Word '2023': tf = 1, idf = 0.5390
    + Document length ratio L_d / avgdl = 10 / 11.00 = 0.9091
    + Length penalty (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Numerator (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Denominator (tf + k1 * penalty) = 1 + 1.2 * 0.9318 = 2.1182
    + Term score = idf * (n

[('doc3', np.float64(1.4691012344075283)),
 ('doc4', np.float64(1.3165407216036695)),
 ('doc1', np.float64(0.5598161080571258)),
 ('doc2', np.float64(0.0)),
 ('doc5', np.float64(0.0))]

### Observations on Test 2 Failure:
- Similar to TF-IDF, **BM25 still fails completely (score of 0.0000)** when searching for the synonym financial term 'capital expenditures' in Amazon's document.
- **Conclusion:** Although Okapi BM25 optimizes term frequency and document length extremely well, its core remains a keyword matcher (Lexical Matcher). Therefore, it is helpless against the lexical gap.